In [1]:
print('Test')

Test


In [4]:
import os
import glob
import shutil
import yaml

# ===== CONFIG =====
OLD_YAML = "koi_data_4/data.yaml"
NEW_YAML = "data_1class.yaml"
DATASET_ROOT = "koi_data_4"  # ปรับ path ตามจริง
SPLITS = ["train", "valid", "test"]
# ==================

def convert_labels(src_label_dir, dst_label_dir):
    """แปลง class index ทุกตัวเป็น 0"""
    os.makedirs(dst_label_dir, exist_ok=True)
    
    txt_files = glob.glob(os.path.join(src_label_dir, "*.txt"))
    print(f"  Found {len(txt_files)} label files in {src_label_dir}")
    
    for src_path in txt_files:
        filename = os.path.basename(src_path)
        dst_path = os.path.join(dst_label_dir, filename)
        
        with open(src_path, "r") as f:
            lines = f.readlines()
        
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                # parts[0] = class_id → เปลี่ยนเป็น 0
                parts[0] = "0"
                new_lines.append(" ".join(parts) + "\n")
        
        with open(dst_path, "w") as f:
            f.writelines(new_lines)

def update_yaml(old_yaml_path, new_yaml_path):
    """อัปเดต data.yaml ให้เป็น 1 class"""
    with open(old_yaml_path, "r") as f:
        data = yaml.safe_load(f)
    
    data["nc"] = 1
    data["names"] = ["fish"]
    
    with open(new_yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True)
    
    print(f"Saved new YAML → {new_yaml_path}")

# ===== MAIN =====
for split in SPLITS:
    src_labels = os.path.join(DATASET_ROOT, split, "labels")
    dst_labels = os.path.join(DATASET_ROOT, f"{split}_1class", "labels")
    
    if os.path.exists(src_labels):
        print(f"Processing: {split}")
        convert_labels(src_labels, dst_labels)
        
        # Copy images (ไม่ต้องแก้ไข)
        src_images = os.path.join(DATASET_ROOT, split, "images")
        dst_images = os.path.join(DATASET_ROOT, f"{split}_1class", "images")
        if os.path.exists(src_images):
            shutil.copytree(src_images, dst_images, dirs_exist_ok=True)
    else:
        print(f"  Skipping {split} (not found)")

update_yaml(OLD_YAML, NEW_YAML)
print("Done!")

Processing: train
  Found 297 label files in koi_data_4\train\labels
Processing: valid
  Found 24 label files in koi_data_4\valid\labels
  Skipping test (not found)
Saved new YAML → data_1class.yaml
Done!


In [5]:
import cv2
import os
from pathlib import Path

# ===================== CONFIG =====================
IMG_DIR = r"koi_data_4/train/images"
LBL_DIR = r"koi_data_4/train/labels"
# ==================================================

img_files = sorted(Path(IMG_DIR).glob("*.jpg"))

# ============================================================
# Step 1: ลบภาพที่ไม่มี label โดยอัตโนมัติก่อนเลย
# ============================================================
no_label = [p for p in img_files if not (Path(LBL_DIR) / (p.stem + ".txt")).exists()]

if no_label:
    print(f"[!] พบภาพที่ไม่มี label {len(no_label)} ไฟล์ → ลบออกอัตโนมัติ")
    for p in no_label:
        os.remove(p)
        print(f"    [deleted] {p.name}")
else:
    print("[OK] ไม่มีภาพที่ไม่มี label")

# โหลดลิสต์ใหม่หลังลบ
img_files = sorted(Path(IMG_DIR).glob("*.jpg"))
print(f"[OK] เหลือภาพทั้งหมด {len(img_files)} ไฟล์\n")

# ============================================================
# Step 2: Preview ตรวจสอบ bounding box ทีละภาพ
# ============================================================
print("Preview: Space/อื่นๆ = ถัดไป  |  D = ลบภาพนี้  |  ESC = ออก\n")

for i, img_path in enumerate(img_files):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    h, w = img.shape[:2]

    lbl_path = Path(LBL_DIR) / (img_path.stem + ".txt")
    box_count = 0

    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, cx, cy, bw, bh = map(float, parts)
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            x2 = int((cx + bw / 2) * w)
            y2 = int((cy + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 80), 2)
            cv2.putText(img, "fish", (x1, max(y1 - 6, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 80), 1)
            box_count += 1

    # Header
    cv2.rectangle(img, (0, 0), (w, 38), (20, 20, 20), -1)
    info = f"{i+1}/{len(img_files)}  |  {img_path.name}  |  Boxes: {box_count}  |  [Space]=Next  [D]=Delete  [ESC]=Quit"
    cv2.putText(img, info, (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1)

    cv2.imshow("Label Preview", img)
    key = cv2.waitKey(0) & 0xFF

    if key == 27:   # ESC
        print("ออกจาก preview")
        break
    elif key == ord('d'):
        os.remove(img_path)
        os.remove(lbl_path)
        print(f"  [deleted] {img_path.name}")

cv2.destroyAllWindows()
remaining = len(list(Path(IMG_DIR).glob("*.jpg")))
print(f"\n[OK] เสร็จ! เหลือภาพทั้งหมด {remaining} ไฟล์")

[OK] ไม่มีภาพที่ไม่มี label
[OK] เหลือภาพทั้งหมด 297 ไฟล์

Preview: Space/อื่นๆ = ถัดไป  |  D = ลบภาพนี้  |  ESC = ออก


[OK] เสร็จ! เหลือภาพทั้งหมด 297 ไฟล์
